# SatQuery — Qwen2.5-VL QLoRA training on Kaggle (Unsloth) — v2

Fine-tunes **Qwen2.5-VL-3B-Instruct** with QLoRA on the BigEarthNet VQA dataset
[`knayamket/bigearthnet-vqa`](https://www.kaggle.com/datasets/knayamket/bigearthnet-vqa).
Self-contained — no repo clone needed. Saves a checkpoint **every 10 minutes**
(and every 50 steps) and **auto-resumes** from the latest checkpoint if the
session dies.

## v2 changes (Sept 12 2026) — fixes the "one-word answer" collapse

The v1 adapter (12k samples · 1 epoch · LR 1e-4 · r=16) trained end-to-end
successfully but collapsed language: every question got a single-token reply
("presence", "no", "area"). Diagnosed via `ml/tests/compare_base_vs_adapter.py`
on branch `feat/local-vlm-serve`. Root causes:

  * Training data was almost entirely short-answer BEN VQA — never saw a
    descriptive/caption response, so the LM head learned "short reply = safe"
  * LR was too aggressive (1e-4) — overwrote the base's language modelling
  * MLP layers were fine-tuned (`finetune_mlp_modules=True`) — that layer
    holds most factual/language priors; touching it collapses generation

v2 fixes each of those:

  * **Data augmentation**: every multi-label answer is also emitted as a
    caption ("This Sentinel-1 scene shows broad-leaved forest and ...").
    Roughly one in three rows gains a descriptive variant.
  * **LR halved: 1e-4 → 5e-5** — preserves base fluency
  * **MLP frozen** (`finetune_mlp_modules=False`) — only attention gets
    the LoRA, which is the "task adapter" pattern from the original QLoRA
    paper
  * **More capacity: r=32, alpha=64** — the richer objective needs room
  * **MAX_SAMPLES 12k → 20k, EPOCHS 1 → 2** — ~3× the gradient updates
    but still fits inside one Kaggle T4 session (~7-9 h)

## Notebook settings (do this first)

1. **Settings → Accelerator → GPU T4** (or P100)
2. **Settings → Internet → On** (needed to download the base model)
3. **Settings → Persistence → Files** (keeps `/kaggle/working` between sessions — this is how resume works)
4. **Add Input →** search `bigearthnet-vqa` (by knayamket) and attach it
5. On a later session, if you backed up checkpoints as a dataset, also attach `ben-lora-checkpoints` and set `RESUME_DIR` in the config cell

Then just **Run All**. The notebook indexes `train.jsonl` by byte offset and
trains on **12,000 random rows** (not the whole ~1 GB file — that OOMs Kaggle
CPU RAM). Raise `MAX_SAMPLES` in the config cell only after a run finishes cleanly.

In [ ]:
# Cell 1 — install Unsloth (takes a couple of minutes)
%pip install -q unsloth

In [ ]:
# Cell 2 — config + locate the dataset
from pathlib import Path

# ---- Dataset (attached via Add Input) ----
DATASET_SLUG = "bigearthnet-vqa"  # kaggle.com/datasets/knayamket/bigearthnet-vqa

# ---- Output / resume ----
# v2 writes to a separate folder so a v1 adapter you already downloaded stays
# untouched. If you want to continue a v2 run from a checkpoint dataset, set
# RESUME_DIR to that Kaggle input path.
OUT = Path("/kaggle/working/ben-lora-v2")
RESUME_DIR = None  # later sessions: Path("/kaggle/input/ben-lora-v2-checkpoints")

# ---- Training knobs (v2 — see the intro markdown cell for rationale) ----
MODEL_NAME = "unsloth/Qwen2.5-VL-3B-Instruct-bnb-4bit"
EPOCHS = 2.0
MAX_STEPS = None      # set e.g. 30 for a quick smoke test
# train.jsonl is ~1 GB. Loading it all into RAM kills Kaggle CPU.
# 20k random rows + ~30% caption augmentation ≈ 25k effective samples,
# one full T4 session (~7-9 h). Raise only after a full run finishes cleanly.
MAX_SAMPLES = 20_000
LR = 5e-5             # v1 was 1e-4 and clobbered the base LM's fluency
BATCH_SIZE = 1
GRAD_ACCUM = 8        # effective batch = 8; smoother gradients, still Kaggle-safe
SAVE_EVERY_MINUTES = 10.0
SAVE_STEPS = 50
SAVE_TOTAL_LIMIT = 3
SEED = 13

# ---- LoRA capacity (v2) ----
LORA_R = 32           # v1 = 16
LORA_ALPHA = 64       # v1 = 32
LORA_DROPOUT = 0.1    # v1 = 0.05 — more regularisation with the bigger rank

# ---- Data augmentation ----
# When >0, every N-th multi-label row also emits a "describe" example whose
# answer is a natural-language caption. Cures the v1 one-word-answer collapse.
CAPTION_AUGMENT_RATE = 0.30

SYSTEM = (
    "You are SatQuery, an assistant for satellite and aerial imagery. "
    "Answer questions about optical and SAR remote sensing scenes "
    "concisely and factually."
)


def find_data_root() -> Path:
    """Find the folder holding train.jsonl — root or one level down,
    depending on how the dataset was uploaded."""
    bases = [Path("/kaggle/input") / DATASET_SLUG]
    if not bases[0].exists():
        # Slug mismatch? Fall back to scanning every attached input.
        bases = [p for p in Path("/kaggle/input").iterdir() if p.is_dir()]
    for base in bases:
        if (base / "train.jsonl").exists():
            return base
        for child in sorted(base.iterdir()):
            if child.is_dir() and (child / "train.jsonl").exists():
                return child
    raise SystemExit(
        "train.jsonl not found under /kaggle/input. "
        "Add Input -> knayamket/bigearthnet-vqa and re-run."
    )


DATA_ROOT = find_data_root()
print("DATA_ROOT =", DATA_ROOT)
print("contents  =", sorted(p.name for p in DATA_ROOT.iterdir())[:10])

In [ ]:
# Cell 3 — sample a Kaggle-safe slice of train.jsonl (do NOT json.loads the whole 1 GB file)
import json
import random

DEFAULT_MAX_SAMPLES = 12_000


def index_jsonl_offsets(path: Path) -> list[int]:
    """Byte offsets of non-empty lines. Does not parse JSON."""
    size_gb = path.stat().st_size / 1e9
    print(f"indexing {path} ({size_gb:.2f} GB) — not loading it into RAM ...", flush=True)
    offsets = []
    with path.open("rb") as fh:
        pos = 0
        for raw in fh:
            if raw.strip():
                offsets.append(pos)
            pos += len(raw)
            if offsets and len(offsets) % 1_000_000 == 0:
                print(f"  indexed {len(offsets):,} rows", flush=True)
    print(f"indexed {len(offsets):,} jsonl rows", flush=True)
    return offsets


# v2 caption augmentation — turn "which classes are present" answers
# ("broad-leaved forest, mixed forest") into descriptive captions so the
# adapter also learns to produce verbose replies.
DESCRIBE_TEMPLATES = [
    "Describe this remote sensing image in detail.",
    "What does this satellite patch show?",
    "Summarise the land cover in this scene.",
    "Provide a natural-language caption for this image.",
]


def _to_caption(labels: list[str], sensor: str) -> str:
    labels = [l for l in labels if l]
    if not labels:
        return f"This {sensor} scene has no clearly dominant land-cover class."
    if len(labels) == 1:
        return f"This {sensor} scene shows {labels[0]}."
    if len(labels) == 2:
        return f"This {sensor} scene shows {labels[0]} and {labels[1]}."
    head = ", ".join(labels[:-1])
    return f"This {sensor} scene shows {head}, and {labels[-1]}."


def _maybe_caption_example(row: dict, rng: random.Random) -> dict | None:
    """Emit a `describe` variant iff the answer is a comma-separated multi-label."""
    ans = row.get("answer", "").strip()
    if "," not in ans or rng.random() >= CAPTION_AUGMENT_RATE:
        return None
    labels = [x.strip() for x in ans.split(",") if x.strip()]
    if len(labels) < 2:
        return None
    sensor = "Sentinel-1 SAR" if row.get("image", "").startswith("images_s1") else "Sentinel-2 optical"
    return {
        "question": rng.choice(DESCRIBE_TEMPLATES),
        "answer": _to_caption(labels, sensor),
        "image": row["image"],
    }


def _row_to_message(row: dict, data_root: Path) -> dict:
    return {
        "messages": [
            {
                "role": "user",
                "content": [
                    {"type": "text", "text": SYSTEM + "\n\n" + row["question"]},
                    {"type": "image", "image": str(data_root / row["image"])},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": row["answer"]}],
            },
        ]
    }


def load_dataset(data_root: Path, max_samples=DEFAULT_MAX_SAMPLES, seed=13):
    train_jsonl = data_root / "train.jsonl"
    offsets = index_jsonl_offsets(train_jsonl)
    total = len(offsets)
    if not total:
        raise SystemExit(f"{train_jsonl} is empty")

    cap = DEFAULT_MAX_SAMPLES if max_samples is None else max_samples
    if cap > 0 and total > cap:
        rng = random.Random(seed)
        offsets = rng.sample(offsets, cap)
        print(f"sampling {len(offsets):,} / {total:,} rows (seed={seed})", flush=True)
    else:
        print(f"using all {total:,} rows", flush=True)

    dataset, skipped, augmented = [], 0, 0
    rng_aug = random.Random(seed + 1)
    with train_jsonl.open("rb") as fh:
        for pos in offsets:
            fh.seek(pos)
            try:
                row = json.loads(fh.readline())
            except json.JSONDecodeError:
                skipped += 1
                continue
            dataset.append(_row_to_message(row, data_root))
            aug = _maybe_caption_example(row, rng_aug)
            if aug is not None:
                dataset.append(_row_to_message(aug, data_root))
                augmented += 1

    if skipped:
        print(f"warning: skipped {skipped} unreadable rows")
    if not dataset:
        raise SystemExit("No usable training samples")
    print(
        f"usable samples: {len(dataset):,}  (base rows: {len(dataset) - augmented:,}, "
        f"caption-augmented: {augmented:,})",
        flush=True,
    )
    return dataset


dataset = load_dataset(DATA_ROOT, MAX_SAMPLES, SEED)

In [ ]:
# Cell 4 — checkpoint helpers (timed saves + auto-resume)
import shutil
import time

from transformers import TrainerCallback


class TimedCheckpointCallback(TrainerCallback):
    """Force a checkpoint every `every_minutes` wall-clock minutes."""

    def __init__(self, every_minutes: float = 10.0):
        self.every_seconds = max(60.0, every_minutes * 60.0)
        self._last = time.monotonic()

    def on_step_end(self, args, state, control, **kwargs):
        now = time.monotonic()
        if now - self._last >= self.every_seconds:
            control.should_save = True
            self._last = now
            print(
                f"\n[checkpoint] timed save at step {state.global_step} "
                f"(every {self.every_seconds / 60:.0f} min)\n",
                flush=True,
            )
        return control


def find_latest_checkpoint(roots):
    candidates = []
    for root in roots:
        if not root or not Path(root).exists():
            continue
        root = Path(root)
        candidates.extend(p for p in root.glob("checkpoint-*") if p.is_dir())
        candidates.extend(p for p in root.glob("**/checkpoint-*") if p.is_dir())
    if not candidates:
        return None

    def step_num(path):
        try:
            return int(path.name.split("-")[-1])
        except ValueError:
            return -1

    return max(candidates, key=step_num)


def seed_checkpoints_from_resume(resume_dir, out_dir: Path):
    """Copy uploaded checkpoints into OUT so the Trainer can resume locally."""
    if resume_dir is None or not Path(resume_dir).exists():
        return
    resume_dir = Path(resume_dir)
    src_ckpts = list(resume_dir.glob("checkpoint-*")) or list(resume_dir.glob("**/checkpoint-*"))
    if not src_ckpts:
        print(f"no checkpoint-* under {resume_dir}")
        return
    out_dir.mkdir(parents=True, exist_ok=True)
    for src in src_ckpts:
        if not src.is_dir():
            continue
        dest = out_dir / src.name
        if dest.exists():
            continue
        print(f"seeding checkpoint -> {dest}")
        shutil.copytree(src, dest)

In [ ]:
# Cell 5 — train (safe to re-run: it resumes from the latest checkpoint)
import os

os.environ.setdefault("HF_HUB_DISABLE_TELEMETRY", "1")

from unsloth import FastVisionModel
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTConfig, SFTTrainer

OUT.mkdir(parents=True, exist_ok=True)
seed_checkpoints_from_resume(RESUME_DIR, OUT)

resume_path = find_latest_checkpoint([OUT, RESUME_DIR] if RESUME_DIR else [OUT])
print(f"RESUMING from {resume_path}" if resume_path else "No checkpoint found — starting a new run")

model, tokenizer = FastVisionModel.from_pretrained(
    MODEL_NAME,
    load_in_4bit=True,
    use_gradient_checkpointing="unsloth",
)
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision_layers=False,
    finetune_language_layers=True,
    finetune_attention_modules=True,
    # v2: MLP frozen. In v1 (finetune_mlp_modules=True) the LoRA rewrote the
    # feed-forward layers where Qwen stores most of its factual and language
    # priors, which is what caused the one-word-answer collapse. Keeping only
    # attention adapters gives us "task adaptation" without "language erasure".
    finetune_mlp_modules=False,
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    random_state=SEED,
)
FastVisionModel.for_training(model)

sft_kwargs = dict(
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    warmup_ratio=0.03,
    learning_rate=LR,
    logging_steps=10,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=SEED,
    output_dir=str(OUT),
    report_to="none",
    remove_unused_columns=False,
    dataset_text_field="",
    dataset_kwargs={"skip_prepare_dataset": True},
    max_seq_length=2048,
    save_strategy="steps",
    save_steps=SAVE_STEPS,
    save_total_limit=SAVE_TOTAL_LIMIT,
    load_best_model_at_end=False,
)
if MAX_STEPS:
    sft_kwargs["max_steps"] = MAX_STEPS
else:
    sft_kwargs["num_train_epochs"] = EPOCHS

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=dataset,
    args=SFTConfig(**sft_kwargs),
    callbacks=[TimedCheckpointCallback(SAVE_EVERY_MINUTES)],
)

trainer.train(resume_from_checkpoint=str(resume_path) if resume_path else None)

adapter_dir = OUT / "adapter"
model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))
print(f"\nFinal adapter -> {adapter_dir}")

In [ ]:
# Cell 6 — backup checkpoints (run anytime, especially before the session dies)
# Download the zip from Output, upload it as a Kaggle Dataset (e.g. ben-lora-v2-checkpoints),
# then next session: Add Input -> that dataset and set RESUME_DIR in Cell 2.
!cd /kaggle/working && zip -q -r ben-lora-v2-checkpoints.zip ben-lora-v2
print("Download ben-lora-v2-checkpoints.zip from the Output tab")

In [ ]:
# Cell 7 — final adapter zip (when training finishes)
# This zip is what ml/serve.py needs on the inference machine.
!cd /kaggle/working && zip -q -r ben-lora-v2-adapter.zip ben-lora-v2/adapter
print("Download ben-lora-v2-adapter.zip from the Output tab")